In [ ]:
# The main reason behind using a memory system is because 
# every LLM call is a unique request, and it doesn't store any state.
# This is why you constantly need to give context to your LLM.

# for this you store the conversation and inject it with the prompt in each request
# which ultimately gives the LLM context on the conversation, make the responses more to the point and wholesome.

# To store these context messages, you can either store them in-memory or in a database.
# in-memory generally fails in production, hence database is preferred for production.

# There are various methods to use this memory system.
# 1. ConverstionBufferMemory which is now become old. 
#    Instead we use ChatMessageHistory + RunnableMessageHistory 
# This automatically stores all the conversations during a session. Hence, its more prone to context overflow,
# making the model to either hallucinate or loose the context. 
# 2. WindowMemory
#    This only store N number of messages at a time and if that limit exceeds, the older messages get trimmed.
# 3. ConversationSummaryBufferMemory
#    This allows you to summarize the previous conversation if the message history exceed a limit of tokens.
# The challenge faced here is, model takes more time to respond everytime the message history is filled up with set limited tokens.
# Inference becomes occassionally slower, as model has to summarize + return the inference.
# 4. ConversationTokenBufferMemory
#    This is similar to WindowMemory, but instead of messages, it makes the limits on tokens.
# Reason being: some messages may have more tokens, some may have lesser, where the chance of filling the context window still remains.
# Hence, you trim older tokens when the memory already has filled up with the token limit.


# IMPLEMENTATION:


In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model = "gemini-flash-latest", temperature = 0.7)
prompt = ChatPromptTemplate(
    [
        ("system", "You are a helpful AI Engineer Assistant."),
        MessagesPlaceholder(variable_name = "chat_message_history"),
        ("human", "{input}")
    ]
)

in_memory = {}

from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id in in_memory:
        return in_memory[session_id]
    else:
        in_memory[session_id] = ChatMessageHistory()

# Wrapping the chain with this ChatMessageHistory.

context_aware_chain = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key= "input",
    history_message_key= "chat_message_history",
)

config = {"configurable": {"session_id": "user_123"}}

response1 = context_aware_chain.invoke({"input": "My Name is Harsh."}, config = config)
print(response1)

response2 = context_aware_chain.invoke({"input":"What is my name?"}, config = config)
print(response2)